# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krashishkr008-ghg/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two Paper Findings + My Methodology Questions

### Finding 1
The paper reports that search ranking signals are associated with improvements in content visibility.

**Methodology question:**  
How was the label defined, and was the same definition used consistently across all clients? This helps determine whether the reported improvement is measured fairly.

---

### Finding 2
The paper reports that machine learning can identify content that benefits from optimization.

**Methodology question:**  
Was the validation performed using an honest grouped or time-aware split? This helps verify that the reported performance generalizes to new data rather than memorizing existing examples.

In [10]:
print("Finding 1 reviewed ✔")
print("Finding 2 reviewed ✔")


Finding 1 reviewed ✔
Finding 2 reviewed ✔


## 2. My Model Under an Honest Split (Before/After)

I re-ran the Week-5 Random Forest model using a grouped train/test split based on client_id.

Compared with the earlier model, this grouped split is more honest because the same client does not appear in both the training and testing data.

The grouped validation gives a more realistic estimate of how the model performs on unseen clients.

The comparison is intended for decision-support and uses observed model performance.

In [11]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Keep required columns
df = df[["search_volume", "avg_position", "ctr", "client_id"]].dropna()

# Features and target
X = df[["search_volume", "avg_position"]]
y = df["ctr"]
groups = df["client_id"]

# Honest grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

# Before vs After table
comparison = pd.DataFrame({
    "Validation": ["Previous Model", "Grouped Split"],
    "MAE": [0.6032, round(mae, 4)],
    "R² Score": [-1.3613, round(r2, 4)]
})

print("Before vs After Validation")
display(comparison)

Before vs After Validation


,Validation,MAE,R² Score
0,Previous Model,0.6032,-1.3613
1,Grouped Split,0.5093,-0.3700


## 3. Leakage Audit

I reviewed the final feature set used in my model.

Features used:
- search_volume
- avg_position

Target:
- ctr

The model does not use the target variable as an input feature. It also does not use future-window information, product flags, client names, URLs, or private queries.

Based on this review, I did not observe evidence of feature leakage. The feature set is appropriate for decision-support.

In [12]:
# Leakage audit

features = ["search_volume", "avg_position"]
target = "ctr"

print("Final Features:")
for feature in features:
    print("-", feature)

print("\nTarget:")
print("-", target)

print("\nLeakage Audit Results")
print("✓ Target (ctr) is not used as an input feature.")
print("✓ No future-window information is used.")
print("✓ No product flags are used.")
print("✓ No client names, URLs, or private queries are used.")
print("✓ No evidence of feature leakage was observed.")

Final Features:
- search_volume
- avg_position

Target:
- ctr

Leakage Audit Results
✓ Target (ctr) is not used as an input feature.
✓ No future-window information is used.
✓ No product flags are used.
✓ No client names, URLs, or private queries are used.
✓ No evidence of feature leakage was observed.


## 4. Claim Rewrite

### Original claim
The Random Forest model predicts CTR better than the baseline.

### Revised claim
In this analysis, the grouped validation measured different performance between the Random Forest model and the Week-4 baseline. The observed results suggest that the grouped validation provides a more realistic estimate of performance on unseen clients. These findings are directional and intended for decision-support rather than proving that one model is universally better.

In [13]:
print("Original Claim:")
print("The Random Forest model predicts CTR better than the baseline.")

print("\nRewritten Claim:")
print("The grouped validation measured different performance between the Random Forest model and the Week-4 baseline.")
print("The observed results are directional and intended for decision-support.")

Original Claim:
The Random Forest model predicts CTR better than the baseline.

Rewritten Claim:
The grouped validation measured different performance between the Random Forest model and the Week-4 baseline.
The observed results are directional and intended for decision-support.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card.